# Õppetund 01 - Sissejuhatus tehisintellekti agentidesse

Tere tulemast esimesse õppetundi kursusel **AI Agents for Beginners**!

**Tehisintellekti agent** on programm, mis kasutab suurt keelemudelit (LLM) oma arutlusmasinana ja saab reaalses maailmas teha *tegevusi* — kutsuda API-sid, otsida andmebaasidest või käivitada koodi — eesmärgiga täita kasutaja ülesandeid.

Selles märkmelehes ehitad oma esimese agendi: **reisibüroo agendi**, kes soovitab puhkuse sihtkohti. Samal ajal õpid kuidas:

1. Ühendada Azure AI Foundry Agent Service'iga, kasutades **Microsoft Agent Frameworki**.
2. Anda agendile **tööriist** — lihtne Python funktsioon, mida ta saab kutsuda.
3. Käivitada agent ja uurida selle vastust.
4. Voogesitada agendi vastust token-tokeni haaval.


## Seadistamine

Enne selle märkmiku käivitamist veenduge, et teil on:

1. **Azure AI Foundry projekt**, kus on juurutatud vestlusmudel (nt `gpt-4o-mini`).
2. **Sisselogitud Azure CLI-ga** — käivitage terminalis käsk `az login`.
3. **Määratud vajalikud keskkonnamuutujad:**
   - `AZURE_AI_PROJECT_ENDPOINT` — teie Azure AI Foundry projekti lõpp-punkt.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — teie juurutatud mudeli nimi.

Järgmine lahter installib vajaminevad Python paketid.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Esimese agendi loomine

Agendi jaoks on vaja kahte asja:

- **Juhised**, mis ütleksid, *kes ta on* ja *kuidas käituda* (süsteemi prompt).
- **Tööriistad** — Python funktsioonid, mis on kaunistatud `@tool` ja mida agent saab kutsuda info saamiseks või toimingute tegemiseks.

Allpool määratleme lihtsa tööriista, mis tagastab nimekirja populaarsetest puhkuse sihtkohtadest. Agent kasutab seda tööriista, kui kasutaja palub reisisoovitusi.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Voogedastuse vastused

Veelgi interaktiivsema kogemuse jaoks saate agendi vastuse **voogedastada**. Täisvastuse ootamise asemel edastab agent tekstitükke, mida ta genereerib. See on eriti kasulik vestlusliidestes, kus soovite väljundit reaalajas kuvada.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Kokkuvõte

Selles õppetükis õppisite, kuidas:

- **Luua pakkuja**, mis ühendub Azure AI Foundry Agent teenusega kasutades `AzureAIProjectAgentProvider`.
- **Määratleda tööriist** `@tool` dekoratsiooni abil, et agent saaks kutsuda teie Python'i funktsioone.
- **Käivita agent** kasutaja sõnumiga ja prindi selle vastus.
- **Voogesitada vastuseid** reaalajas väljundi saamiseks.

Järgmises õppetükis uurime agentide raamistikke põhjalikumalt ning õpime, kuidas anda agentidele võimsamaid tööriistu ja mitmetasandilise mõtlemise võimekust.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Lahtiütlus**:
See dokument on tõlgitud kasutades tehisintellektil põhinevat tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi püüame täpsust, palun arvestage, et automatiseeritud tõlked võivad sisaldada vigu või ebatäpsusi. Originaaldokument oma emakeeles tuleks pidada autoriteetseks allikaks. Olulise info puhul soovitatakse kasutada professionaalset inimtõlget. Me ei vastuta selle tõlke kasutamisest tulenevate arusaamatuste või väärinterpretatsioonide eest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
